In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from jubilee_controller import JubileeMotionController
import matplotlib.pyplot as plt
import time
import numpy as np 
from pynput import keyboard
import cv2
import lgpio
from datetime import datetime
from camera_controller import camera_tool
from email_sender import send_email
from image_processing import detect_circle
from gripper_controller import Gripper
from micropipette_controller import Micropipette
from magnetic_stirrer_controll import Agitador

In [3]:
jubilee = JubileeMotionController()

In [4]:
jubilee.home_all(mesh_mode_z=False)
jubilee.protect_tools(on=True)
jubilee.move_xyz_absolute(z=50)

In [5]:
pipeta = Micropipette(jubilee,parking_position_xy=(136.5,16),move_velocity=3000)

In [14]:
pipeta.uninstall()

In [7]:
agitador = Agitador()

Agitador configurado no GPIO6.


In [8]:
jubilee.protect_tools(True)

In [9]:
jubilee.gcode("M208 Z0:320")

''

In [ ]:
agitador.desligar()

In [10]:
altura_segura_1 = 320
altura_segura_2 = 320
velocidade_movimento_xy = 6000
posicao_descarte = (105,240,270)

In [11]:
def acoplar_ponteira(altura_segura, posicao_caixa_ponteiras):
    jubilee.move_xyz_absolute(z=altura_segura)
    jubilee.move_xyz_absolute(posicao_caixa_ponteiras[0], posicao_caixa_ponteiras[1], velocity=velocidade_movimento_xy)
    jubilee.move_xyz_absolute(z=posicao_caixa_ponteiras[2], velocity=500)
    jubilee.move_xyz_absolute(z=altura_segura)
    posicao_caixa_ponteiras[0] -= 10



def descarte_ponteira(altura_segura,posicao_descarte):
    jubilee.move_xyz_absolute(z=altura_segura)
    jubilee.move_xyz_absolute(posicao_descarte[0], posicao_descarte[1], velocity=velocidade_movimento_xy)
    jubilee.move_xyz_absolute(z=posicao_descarte[2], velocity=1000)
    pipeta.eject_tip()


def pipetar_liquido(origem, destino, volume_ul, altura_segura,velocidade_movimento_xy_z, velocidade_dispensacao=300):
    ciclos = int(volume_ul // 1000)
    restante = volume_ul % 1000

    def transferir(volume):
        jubilee.move_xyz_absolute(z=altura_segura,velocity = velocidade_movimento_xy_z[1])
        pipeta.press(volume)
        jubilee.move_xyz_absolute(origem[0], origem[1], velocity=velocidade_movimento_xy_z[0])
        jubilee.move_xyz_absolute(z=origem[2], velocity=velocidade_movimento_xy_z[1])
        pipeta.aspirate()
        jubilee.move_xyz_absolute(z=altura_segura)
        jubilee.move_xyz_absolute(destino[0], destino[1], velocity=velocidade_movimento_xy_z[0])
        jubilee.move_xyz_absolute(z=destino[2],velocity=velocidade_movimento_xy_z[0])
        pipeta.dispense(velocity=velocidade_dispensacao)

    for _ in range(ciclos):
        transferir(1000)

    if restante > 0:
        transferir(restante)


<p align='justify'>
Inicialmente, 1 mL de nitratos de lantanídeos na proporção 1:0,3 (Gd:Eu), 
com concentrações de 0,2 M de 
<em>Gd(NO<sub>3</sub>)<sub>3</sub>·6H<sub>2</sub>O</em> e 0,06 M de 
<em>Eu(NO<sub>3</sub>)<sub>3</sub>·6H<sub>2</sub>O</em> 
(Sigma-Aldrich), foram adicionados a um balão de fundo redondo acoplado a um condensador. 
Em seguida, 2 mL de 
<em>citrato de sódio</em> (Citrato de Sódio Tribásico Anidro PA, Sigma-Aldrich) 
0,2 M foram adicionados gradualmente ao longo de 3 minutos, sob agitação magnética. 
Após a estabilização da temperatura do sistema a 90&nbsp;°C, utilizando banho de glicerina, 
4 mL de 
<em>NaF</em> (Sigma-Aldrich) 1 M foram adicionados ao sistema, 
que foi mantido sob agitação contínua e temperatura constante de 90&nbsp;°C por 2 horas.
</p>


In [ ]:
#jubilee.move_xyz_absolute(243,161,230)

In [ ]:
jubilee.move_xyz_absolute(z=320,velocity=1600)


In [12]:
posicao_balao = (267,161,305)
posicao_caixa_ponteiras = [144.9,312,160]

nitrato_lantanideos = (31,191,171)
#citrato_sodio = (141.0,131.0,153.00)
citrato_sodio = (267,281.0,238.00)



In [ ]:
agitador.ligar()

#Acoplar Primeira Ponteira
acoplar_ponteira(altura_segura=altura_segura_1,posicao_caixa_ponteiras=posicao_caixa_ponteiras)

#Pipeta 1000ul de ácido cloroáurico
pipetar_liquido(nitrato_lantanideos,posicao_balao,1000,altura_segura_2,velocidade_movimento_xy_z=[velocidade_movimento_xy,600],velocidade_dispensacao=300)

#Remover ponteira
descarte_ponteira(altura_segura_1,posicao_descarte)

#Acoplar Segunda Ponteira
acoplar_ponteira(altura_segura=altura_segura_1,posicao_caixa_ponteiras=posicao_caixa_ponteiras)

#Pipeta 500ul de borohidreto de sódio
pipetar_liquido(citrato_sodio,posicao_balao,1000,altura_segura_2,velocidade_movimento_xy_z=[10000,1600],velocidade_dispensacao=13)
pipetar_liquido(citrato_sodio,posicao_balao,1000,altura_segura_2,velocidade_movimento_xy_z=[10000,1600],velocidade_dispensacao=13)

#Remover ponteira
descarte_ponteira(altura_segura_1,posicao_descarte)
time.sleep(600)

#Desativar Agitador
agitador.desligar()
agitador.fechar()



In [13]:
pipetar_liquido(citrato_sodio,posicao_balao,2000,altura_segura_2,velocidade_movimento_xy_z=[18000,1600],velocidade_dispensacao=13)